# 2: Entity Resolution (Part 2) & Sample Construction

**Objective**
Combine the raw Orbis exports for both ETS and non-ETS universes into two clean, non-overlapping company samples (final_ets_sample.csv, final_non_ets_sample.csv) that meet the data availability and listing criteria required for the study period (2018–2024).

**Strategy**
The core challenge is twofold:
* correctly consolidating ETS installations under their **listed parent entities** using GUO logic, since installations are registered at the operating subsidiary level but stock/financial data exists at the listed parent level
* constructing a non-ETS control group that is free of carbon-regulated entities and comparable in terms of listing characteristics, so that any return differences can be attributed to carbon exposure rather than confounding factors like exchange, size, or listing era.


Filtering is applied programmatically and sequentially, with each stage's row count logged, so that attrition is auditable and defensible in the methodology write-up.

### Load Listing Dataframes and Schema Validation

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

In [2]:
# ── path setup ────────────────────────────────────────────────
sys.path.append(str(Path().resolve().parent))
from src.config import DATA_RAW, DATA_PROCESSED

# ── reusable loader ───────────────────────────────────────────
def load_processed(filename, **kwargs):
    """Load a processed CSV from data/raw folder."""
    return pd.read_csv(DATA_PROCESSED / filename, low_memory=False, **kwargs)

def load_raw(filename, **kwargs):
    """Load a raw CSV from data/raw folder."""
    return pd.read_csv(DATA_RAW / filename, low_memory=False, **kwargs)

# ── reusable profiler ─────────────────────────────────────────
def quick_profile(df, name="DataFrame"):
    """Print shape, dtypes, missing rates."""
    print(f"\n{'='*50}")
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"{'='*50}")

    print("\nColumns + dtypes:")
    print(df.dtypes.to_string())

    print("\nMissing value rates (%):")
    missing = (df.isnull().sum() / len(df) * 100).round(2)
    print(missing[missing > 0].to_string() if missing.any() else "  None")
    print()
    
    return df.head(3)


In [3]:
ets_listing = load_processed('ets_listing_processed.csv')
eu_listing = load_processed('eu_listing_processed.csv')

Some stages of schema validations:
* Deduplicate on bvd_id (Orbis exports sometimes have overlapping boundary rows across parts)
* Log row counts pre/post deduplication
* Confirm identical column schema across both DataFrames
* Standardise data types (dates as datetime, IDs as string to preserve leading zeros)
* Log missing value rates per column for documentation



In [4]:
def de_duplications(listing_df):
    df = listing_df.copy()
    pre_deduplication = len(df)
    print('Number of rows pre-deduplication: ',pre_deduplication)

    duplicated_rows = df[df.duplicated()]
    n_duplicated_rows = df.duplicated().sum()
    print('Number of duplicated rows: ',n_duplicated_rows)

    deduplicated_df = df.drop(duplicated_rows.index)
    print(f'Number of rows post-deduplication: {len(deduplicated_df)}\n')

    return deduplicated_df, duplicated_rows, n_duplicated_rows

In [5]:
ets_listing, ets_duplicated_rows, ets_n_duplicated_rows = de_duplications(ets_listing)
eu_listing, eu_duplicated_rows, eu_n_duplicated_rows = de_duplications(eu_listing)


Number of rows pre-deduplication:  7959
Number of duplicated rows:  1
Number of rows post-deduplication: 7958

Number of rows pre-deduplication:  21664
Number of duplicated rows:  4
Number of rows post-deduplication: 21660



In [6]:
# Confirm that columns and 
# ── reusable profiler ─────────────────────────────────────────
def quick_profile(df, name="DataFrame"):
    """Print shape, dtypes, missing rates."""
    print(f"\n{'='*50}")
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"{'='*50}")

    print("\nColumns + dtypes:")
    print(df.dtypes.to_string())

    print("\nMissing value rates (%):")
    missing = (df.isnull().sum() / len(df) * 100).round(2)
    print(missing[missing > 0].to_string() if missing.any() else "  None")
    print()
    
    return df.head(3)

In [7]:
def standardise_schema(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardise column names, data types, and string formatting.
    - Column names: lowercase, underscored
    - BvD IDs and string IDs: force to string, strip whitespace
    - Date columns: parse to datetime
    - Numeric columns: coerce where appropriate
    """
    # Standardise column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r'[\s/\-]+', '_', regex=True)
        .str.replace(r'[^\w]', '', regex=True)
    )

    # Define ID columns to keep as string
    id_cols = [c for c in df.columns if any(
        kw in c for kw in ['bvd_id', 'bvdid', 'isin', 'ticker',
                            'tax_id', 'trade_register', 'nace']
    )]
    for col in id_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().replace('nan', pd.NA)

    # Define date columns to parse
    date_cols = [c for c in df.columns if any(
        kw in c for kw in ['date', 'ipo', 'delisting', 'incorporation']
    )]
    for col in date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)

    # Remaining object columns: strip whitespace
    obj_cols = df.select_dtypes(include='object').columns
    for col in obj_cols:
        df[col] = df[col].str.strip()

    return df


def validate_schema(df_ets: pd.DataFrame, df_non_ets: pd.DataFrame) -> bool:
    """
    Confirm identical column schema between ETS and non-ETS DataFrames.
    Logs any discrepancies clearly.
    """
    ets_cols = set(df_ets.columns)
    non_ets_cols = set(df_non_ets.columns)

    only_in_ets = ets_cols - non_ets_cols
    only_in_non_ets = non_ets_cols - ets_cols

    print(f"\n{'='*50}")
    print("Schema Validation")
    print(f"{'='*50}")
    print(f"ETS columns:     {len(ets_cols)}")
    print(f"Non-ETS columns: {len(non_ets_cols)}")

    if only_in_ets:
        print(f"\n  [!] Only in ETS:     {sorted(only_in_ets)}")
    if only_in_non_ets:
        print(f"  [!] Only in non-ETS: {sorted(only_in_non_ets)}")
    if not only_in_ets and not only_in_non_ets:
        print("\n  [OK] Schemas are identical.")
        return True

    return False

In [8]:
ets_listing     = standardise_schema(ets_listing)
eu_listing = standardise_schema(eu_listing)

validate_schema(ets_listing, eu_listing)


Schema Validation
ETS columns:     40
Non-ETS columns: 40

  [OK] Schemas are identical.


True

### Normalizing Listing Tables

In [9]:
# profiling listing dataframe and normalizing vat_tax_numbers:
quick_profile(ets_listing)


DataFrame: 7,958 rows × 40 cols

Columns + dtypes:
company_name                                               object
country_iso_code                                           object
nace_rev_2_core_code_4_digits                              object
consolidation_code                                         object
last_avail_year                                           float64
operating_revenue_turnover_th_usd_last_avail_yr            object
number_of_employees_last_avail_yr                          object
country                                                    object
website_address                                            object
bvd_sectors                                                object
bvd_id_number                                              object
trade_register_number                                      object
vat_tax_number                                             object
european_vat_number                                        object
tax_identification_numbe

,company_name,country_iso_code,nace_rev_2_core_code_4_digits,consolidation_code,last_avail_year,operating_revenue_turnover_th_usd_last_avail_yr,number_of_employees_last_avail_yr,country,website_address,bvd_sectors,...,duo_name,duo_ticker_symbol,duo_country_iso_code,ish_name,ish_bvd_id_number,ish_ticker_symbol,ish_country_iso_code,standardized_legal_form,nace_rev_2_core_code_description,nace_rev_2_core_code_4_digits1
0,BAYERISCHE MOTOREN WERKE AG,DE,2910.0,C2,2025.0,157701411.601067,154540,Germany,www.bmwgroup.com,Transport Manufacturing,...,BAYERISCHE MOTOREN WERKE AG,BMW,DE,NaN,<NA>,<NA>,NaN,Public limited companies,Manufacture of motor vehicles,2910.0
1,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,<NA>,NaN,NaN,<NA>,<NA>,NaN,NaN,<NA>,<NA>
2,ALLIANZ SE,DE,6511.0,C2,2025.0,120859295.571804,n.a.,Germany,www.allianz.com,"Banking, Insurance & Financial Services",...,ALLIANZ SE,ALV,DE,NaN,<NA>,<NA>,NaN,Public limited companies,Life insurance,6511.0


In [194]:
quick_profile(eu_listing)


DataFrame: 21,660 rows × 40 cols

Columns + dtypes:
company_name                                               object
country_iso_code                                           object
nace_rev_2_core_code_4_digits                              object
consolidation_code                                         object
last_avail_year                                           float64
operating_revenue_turnover_th_usd_last_avail_yr            object
number_of_employees_last_avail_yr                          object
country                                                    object
website_address                                            object
bvd_sectors                                                object
bvd_id_number                                              object
trade_register_number                                      object
vat_tax_number                                             object
european_vat_number                                        object
tax_identification_numb

,company_name,country_iso_code,nace_rev_2_core_code_4_digits,consolidation_code,last_avail_year,operating_revenue_turnover_th_usd_last_avail_yr,number_of_employees_last_avail_yr,country,website_address,bvd_sectors,...,duo_name,duo_ticker_symbol,duo_country_iso_code,ish_name,ish_bvd_id_number,ish_ticker_symbol,ish_country_iso_code,standardized_legal_form,nace_rev_2_core_code_description,nace_rev_2_core_code_4_digits1
0,SHELL PLC,GB,610.0,C1,2025.0,267418000.0,85000,United Kingdom,www.shell.com,Mining & Extraction,...,SHELL PLC,SHEL,GB,NaN,<NA>,<NA>,NaN,Public limited companies,Extraction of crude petroleum,610.0
1,GLENCORE PLC,GB,2399.0,C1,2025.0,247728000.0,n.a.,United Kingdom,www.glencore.com,"Leather, Stone, Clay & Glass products",...,GLENCORE PLC,GLEN,GB,NaN,<NA>,<NA>,NaN,Public limited companies,Manufacture of other non-metallic mineral prod...,2399.0
2,BP P.L.C.,GB,1920.0,C1,2025.0,189612000.0,93700,United Kingdom,www.bp.com,"Chemicals, Petroleum, Rubber & Plastic",...,BP P.L.C.,BP.,GB,NaN,<NA>,<NA>,NaN,Public limited companies,Manufacture of refined petroleum products,1920.0


Raw Orbis exports contain one row per company per multi-valued attribute (VAT numbers), resulting in continuation null rows for all other fields. Further, ownership data contains significant NULLs, showing consistent null rates across the same group of data.

 Missing values in the raw export are therefore structurally missing (MNAR) rather than reflecting genuine data absence. To eliminate structural missingness and produce an interpretable core entity table, we apply vertical decomposition. The raw DataFrame is split into:
* (1) a core entity table keyed on BvD ID
* (2) a long-format identifier table for tax and regulatory identifiers
* (3) a long-format ownership table capturing GUO, DUO, and ISH relationships. 

Missing values remaining in the core entity table after decomposition are genuinely missing and treated as such in subsequent analysis.

In [10]:
def decompose_core(df: pd.DataFrame, id_col: str = 'bvd_id_number') -> pd.DataFrame:
    """
    Extract core entity table — one row per company.
    Drops continuation null rows (where bvd_id is null) and all
    identifier/ownership columns that are decomposed into separate tables.
    Remaining missingness in this table is genuine, not structural.
    """
    
    id_col = 'bvd_id_number'
    
    #isolate identifier columns & Ownship columns:
    identifier_cols = [
        'vat_tax_number', 'european_vat_number',
        'tax_identification_number_tin', 'lei_legal_entity_identifier',
        'trade_register_number'
    ]
    ownership_cols = [
        'guo_name', 'guo_bvd_id_number', 'guo_country_iso_code', 'guo_ticker_symbol',
        'duo_name', 'duo_bvd_id_number', 'duo_country_iso_code', 'duo_ticker_symbol',
        'ish_name', 'ish_bvd_id_number', 'ish_country_iso_code', 'ish_ticker_symbol'
    ]

    cols_to_drop = [c for c in identifier_cols + ownership_cols if c in df.columns]

    # Drop non-core columns for core df:
    core = (
        df.dropna(subset=[id_col])
          .drop(columns=cols_to_drop)
          .drop_duplicates(subset=[id_col])
          .reset_index(drop=True)
    )

    print(f"\n{'='*50}")
    print("Core Entity Table")
    print(f"{'='*50}")
    
    print(f"  Rows: {len(core):,}")
    print(f"  Columns: {len(core.columns)}")
    print(f"\n  Remaining missing rates (%):")
    missing = (core.isnull().sum() / len(core) * 100).round(2)
    print(missing[missing > 0].to_string() if missing.any() else "  None")

    return core




In [11]:
def decompose_identifiers(df: pd.DataFrame,
                           id_col: str = 'bvd_id_number') -> pd.DataFrame:
    """
    Extract long-format identifier table (one row per BvD ID + identifier type).
    Covers: VAT, European VAT, TIN, LEI, trade register number.
    Rows where identifier_value is null are dropped.
    """

    id_col = 'bvd_id_number'

    # map existing identifier columns in listing table:
    identifier_cols = {
        'vat_tax_number':               'vat_tax_number',
        'european_vat_number':          'european_vat_number',
        'tax_identification_number_tin':'tin', # shorten
        'lei_legal_entity_identifier':  'lei', # shorten
        'trade_register_number':        'trade_register_number'
    }

    # Only use columns that exist in df (contain in available dictionary)
    available = {k: v for k, v in identifier_cols.items() if k in df.columns}

    # Convert wide listing table to long identifier table:
    frames = []
    for col, label in available.items():
        tmp = (
            df[[id_col, col]]
            .dropna(subset=[id_col])   # drop continuation null rows first
            .dropna(subset=[col])      # then drop missing identifiers
            .rename(columns={col: 'identifier_value'})
            .assign(identifier_type=label)
        )
        frames.append(tmp)

    identifier_df = (
        pd.concat(frames, ignore_index=True)
          [[id_col, 'identifier_type', 'identifier_value']]
          .drop_duplicates()
          .reset_index(drop=True)
    )

    print(f"\n{'='*50}")
    print("Identifier Table")
    print(f"{'='*50}")
    print(f"  Total rows:              {len(identifier_df):,}")
    print(f"  Unique BvD IDs:          {identifier_df[id_col].nunique():,}")
    print(f"\n  Rows per identifier type:")
    print(identifier_df['identifier_type'].value_counts().to_string())

    return identifier_df



In [12]:
def decompose_ownership(df: pd.DataFrame,
                        id_col: str = 'bvd_id_number') -> pd.DataFrame:
    """
    Extract long-format ownership table (one row per BvD ID + owner type).
    Covers: GUO, DUO, ISH.
    Rows where all owner fields are null are dropped.
    """
    id_col = 'bvd_id_number'
    
    ownership_groups = {
        'GUO': {
            'name':       'guo_name',
            'bvd_id':     'guo_bvd_id_number',
            'country':    'guo_country_iso_code',
            'ticker':     'guo_ticker_symbol'
        },
        'DUO': {
            'name':       'duo_name',
            'bvd_id':     'duo_bvd_id_number',
            'country':    'duo_country_iso_code',
            'ticker':     'duo_ticker_symbol'
        },
        'ISH': {
            'name':       'ish_name',
            'bvd_id':     'ish_bvd_id_number',
            'country':    'ish_country_iso_code',
            'ticker':     'ish_ticker_symbol'
        }
    }
    
    
    # convert 
    frames = [] # initiate temp ownership

    # loop through each owner type and each column within:
    for owner_type, col_map in ownership_groups.items():
        # Only use columns present in df
        available = {k: v for k, v in col_map.items() if v in df.columns}
        if not available:
            continue
        src_cols = list(available.values())

        rename_map = {v: k for k, v in available.items()}

    # convert each owner-type & column type (wide) to long temp table
        tmp = (
            df[[id_col] + src_cols]
            .dropna(subset=[id_col])          # drop continuation null rows
            .dropna(subset=src_cols, how='all') # drop rows where all owner fields null
            .rename(columns=rename_map)
            .assign(owner_type=owner_type)
        )
        frames.append(tmp)# and append to list

    # 
    ownership_df = (
        pd.concat(frames, ignore_index=True) # create dataframe
          [[id_col, 'owner_type', 'name', 'bvd_id', 'country', 'ticker']]
          .rename(columns={
              'name':   'owner_name',
              'bvd_id': 'owner_bvd_id_number',
              'country':'owner_country',
              'ticker': 'owner_ticker'
          })
          .drop_duplicates()
          .reset_index(drop=True)
    )

    print(f"\n{'='*50}")
    print("Ownership Table")
    print(f"{'='*50}")
    print(f"  Total rows:     {len(ownership_df):,}")
    # print(f"  Unique BvD IDs: {len(ownership_df[id_col].unique()):,}")
    print(f"\n  Rows per owner type:")
    print(ownership_df['owner_type'].value_counts().to_string())

    return ownership_df

In [13]:
# ETS
ets_core       = decompose_core(ets_listing)
ets_identifiers = decompose_identifiers(ets_listing)
ets_ownership  = decompose_ownership(ets_listing)


Core Entity Table
  Rows: 6,803
  Columns: 23

  Remaining missing rates (%):
country_iso_code                     0.01
nace_rev_2_core_code_4_digits        1.59
last_avail_year                      2.25
country                              0.01
website_address                      8.25
bvd_sectors                          2.12
ticker_symbol                       95.35
isin_number                         95.27
date_of_incorporation               11.95
currency                            96.46
type_of_share                       96.46
ipo_date                            97.00
delisting_date                      97.68
nace_rev_2_core_code_description     1.59
nace_rev_2_core_code_4_digits1       1.59

Identifier Table
  Total rows:              25,253
  Unique BvD IDs:          6,789

  Rows per identifier type:
identifier_type
european_vat_number      5890
vat_tax_number           5205
lei                      4984
trade_register_number    4975
tin                      4199

Ownership 

In [14]:
# Non-ETS
eu_core        = decompose_core(eu_listing)
eu_identifiers = decompose_identifiers(eu_listing)
eu_ownership   = decompose_ownership(eu_listing)


Core Entity Table
  Rows: 20,020
  Columns: 23

  Remaining missing rates (%):
nace_rev_2_core_code_4_digits        1.39
last_avail_year                     22.77
website_address                      8.72
bvd_sectors                          1.53
ticker_symbol                       24.53
isin_number                         23.63
date_of_incorporation               58.62
currency                            38.34
type_of_share                       38.34
main_exchange                       19.70
ipo_date                            42.85
delisting_date                      66.20
standardized_legal_form              0.03
nace_rev_2_core_code_description     1.39
nace_rev_2_core_code_4_digits1       1.39

Identifier Table
  Total rows:              55,953
  Unique BvD IDs:          19,200

  Rows per identifier type:
identifier_type
lei                      16305
trade_register_number    11857
european_vat_number      10367
vat_tax_number            9173
tin                       8251

Own

In [15]:
def verify_roundtrip(df_original: pd.DataFrame,
                     core: pd.DataFrame,
                     identifiers: pd.DataFrame,
                     ownership: pd.DataFrame,
                     id_col: str = 'bvd_id_number',
                     name: str = "DataFrame") -> None:
    """
    Verify decomposition roundtrip by rejoining core, identifiers, and ownership
    and comparing shape and values against the original cleaned DataFrame.
    
    Note: compare against the post-standardise, post-dropna(bvd_id) original,
    not the raw export — continuation null rows are intentionally removed.
    """
    id_col = 'bvd_id_number'
    

    # (1) Pivot long identifiers dataframe back to wide:
    id_wide = (
        identifiers
        .pivot_table(index=id_col,
                     columns='identifier_type',
                     values='identifier_value',
                     aggfunc='first')
        .reset_index()
    )
    id_wide.columns.name = None

    # (2) Pivot long ownership dataframe back to wide:
    frames = []
    for owner_type in ownership['owner_type'].unique():
        tmp = (
            ownership[ownership['owner_type'] == owner_type] # subset dataframe for each ownership type
            .drop(columns='owner_type')
            .rename(columns={
                'owner_name':    f'{owner_type.lower()}_name',
                'owner_bvd_id':  f'{owner_type.lower()}_bvd_id_number',
                'owner_country': f'{owner_type.lower()}_country_iso_code',
                'owner_ticker':  f'{owner_type.lower()}_ticker_symbol'
            })
        )
        frames.append(tmp)

    # merge each appended tmp horizontally (outer) on id_col:
    own_wide = frames[0]
    for f in frames[1:]:
        own_wide = own_wide.merge(f, on=id_col, how='outer')

    # Rejoin everything
    reconstructed = (
        core
        .merge(id_wide,  on=id_col, how='left')
        .merge(own_wide, on=id_col, how='left')
    )

    # --- Checks ---
    print(f"\n{'='*50}")
    print(f"Roundtrip Verification — {name}")
    print(f"{'='*50}")

    # 1. Row count
    # Original must be filtered to non-null bvd_id rows first (continuation rows removed)
    df_clean = df_original.dropna(subset=[id_col]).drop_duplicates(subset=[id_col])
    print(f"\n  Row counts:")
    print(f"    Original (post-clean): {len(df_clean):,}")
    print(f"    Reconstructed:         {len(reconstructed):,}")
    print(f"    Match: {'YES' if len(df_clean) == len(reconstructed) else 'NO — investigate'}")

    # 2. BvD ID sets match
    orig_ids  = set(df_clean[id_col])
    recon_ids = set(reconstructed[id_col])
    missing_from_recon = orig_ids - recon_ids
    extra_in_recon     = recon_ids - orig_ids
    print(f"\n  BvD ID set:")
    print(f"    Missing from reconstructed: {len(missing_from_recon)}")
    print(f"    Extra in reconstructed:     {len(extra_in_recon)}")

    # 3. Column coverage
    orig_cols  = set(df_clean.columns)
    recon_cols = set(reconstructed.columns)
    print(f"\n  Column coverage:")
    print(f"    Original:      {len(orig_cols)}")
    print(f"    Reconstructed: {len(recon_cols)}")
    missing_cols = orig_cols - recon_cols
    if missing_cols:
        print(f"    Missing cols:  {sorted(missing_cols)}")
    else:
        print(f"    All original columns present: YES")

    # 4. Spot-check nulls match on core columns
    core_cols = list(core.columns)
    core_cols.remove(id_col)
    orig_nulls  = df_clean[core_cols].isnull().sum()
    recon_nulls = reconstructed[core_cols].isnull().sum()
    mismatch = orig_nulls[orig_nulls != recon_nulls]
    print(f"\n  Null count mismatches on core columns:")
    if mismatch.empty:
        print(f"    None — core columns match exactly")
    else:
        print(mismatch.to_string())

In [16]:
verify_roundtrip(ets_listing,ets_core, ets_identifiers, ets_ownership, name='ETS')


Roundtrip Verification — ETS

  Row counts:
    Original (post-clean): 6,803
    Reconstructed:         6,803
    Match: YES

  BvD ID set:
    Missing from reconstructed: 0
    Extra in reconstructed:     0

  Column coverage:
    Original:      40
    Reconstructed: 40
    Missing cols:  ['duo_bvd_id_number', 'guo_bvd_id_number', 'ish_bvd_id_number', 'lei_legal_entity_identifier', 'tax_identification_number_tin']

  Null count mismatches on core columns:
    None — core columns match exactly


In [17]:
verify_roundtrip(eu_listing, eu_core, eu_identifiers,eu_ownership, name = 'EU')


Roundtrip Verification — EU

  Row counts:
    Original (post-clean): 20,020
    Reconstructed:         20,020
    Match: YES

  BvD ID set:
    Missing from reconstructed: 0
    Extra in reconstructed:     0

  Column coverage:
    Original:      40
    Reconstructed: 40
    Missing cols:  ['duo_bvd_id_number', 'guo_bvd_id_number', 'ish_bvd_id_number', 'lei_legal_entity_identifier', 'tax_identification_number_tin']

  Null count mismatches on core columns:
    None — core columns match exactly


### (2) Mapping Parent Companies Tickers to S&P

In [18]:
# Get all the ownership tickers
ets_self_owned = ets_ownership[ets_ownership['bvd_id_number'] == ets_ownership['owner_bvd_id_number']]
ets_non_selfowned = ets_ownership[ets_ownership['bvd_id_number'] != ets_ownership['owner_bvd_id_number']]


# Get all the non-self-owned companies with tickers:
ets_owner_ticker = ets_non_selfowned[
    ets_non_selfowned['owner_ticker'].notna() &
    ~ets_non_selfowned['owner_ticker'].isin(['-', 'Delisted', ''])
]

ets_ticker_lookup = (
    ets_owner_ticker[['owner_ticker', 'owner_name']]
    .drop_duplicates(subset=['owner_ticker'])
    .reset_index(drop=True)
)

# Generate all variants of tickers:
ets_ticker_lookup['bare_ticker'] = ets_ticker_lookup['owner_ticker'].str.split('.').str[0]
ets_ticker_lookup['space_ticker'] = ets_ticker_lookup['owner_ticker'].str.replace('.', ' ', regex=False)
combined_ets_ticker_lookup = pd.concat([ets_ticker_lookup['bare_ticker'],
                                        ets_ticker_lookup['space_ticker'],
                                        ets_ticker_lookup['owner_ticker']], ignore_index=True).to_frame(name='ticker')
combined_ets_ticker_lookup = combined_ets_ticker_lookup.drop_duplicates()

# Get all the delisted owners with no tickers (why? if delisting is after study year of 2018, still need to consider):
ets_owner_delisted_name = ets_non_selfowned[
    ets_non_selfowned['owner_ticker'] == 'Delisted'
]['owner_name'].unique()

ets_delisted_lookup = (
    ets_non_selfowned[ets_non_selfowned['owner_ticker'] == 'Delisted']
    [['owner_name', 'owner_bvd_id_number', 'owner_country']]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [19]:
# Export for Capital IQ manual lookup
combined_ets_ticker_lookup.to_csv('ciq_ticker_lookup.csv', index=False)
ets_delisted_lookup.to_csv('ciq_delisted_lookup.csv', index=False)

### (2): Map Parents to tickers + installations-parent Resolution


Stage 1  build_ets_parent_map()   — resolve each ETS entity to a listet parent via CIQ ticker matching
Stage 2  flag_delisted_parents()  — rescue entities whose parent delisted during the study window

Output: `ets_parent_map`, exactly one row per ETS bvd_id_number.
── Fixes baked in (all found by running earlier versions) ────────────────
1. all_ids drops null bvd_id — Orbis continuation rows produced a phantom 6,804th entity.
2. country key is `country_iso_code`, not `country`. SP_COUNTRY_CODE is ISO-2; matching full country names against it silently killed self-resolution (self was 23, should be 42).
3. Match passes are country-guarded. The no-country fallback fires only for CIQ tickers unique across countries, so it cannot invent matches.
4. CIQ pool deduped on (SP_TICKER, SP_COUNTRY_CODE) — every merge stays 1:1.
5. Stage 2 selects ONLY from unresolvable_* — never from ~in_portfolio_universe. `non_european` is a decision, not an unresolved state; treating it as one resurrected 8 US/JP parents (KKR, AbbVie...).
6. Stage 2 overwrites ALL parent_* fields from the delisted lookup. Leaving Stage 1 residuals mixed a CIQ-matched ticker with another company's delisting date.
7. "No delisting date" is its own outcome, not lumped into pre-2018; ~37 entities were unknown, not too old.
8. Non-European backstop runs inside the pipeline, applied after both stages. -->

In [20]:
STUDY_START = pd.Timestamp('2018-01-01')

# entity-level column names (confirmed against ets_listing)
CORE_COLS = {
    'ticker':   'ticker_symbol',
    'isin':     'isin_number',
    'name':     'company_name',
    'country':  'country_iso_code',   # ISO-2 — must match SP_COUNTRY_CODE
    'exchange': 'main_exchange',
    'status':   'listing_status',
}

CIQ_COLS = ['SP_ENTITY_NAME', 'SP_ENTITY_ID', 'SP_TICKER', 'SP_EXCHANGE',
            'SP_ISIN', 'SP_LEI', 'SP_COMPANY_TYPE', 'SP_COMPANY_STATUS',
            'SP_COUNTRY_CODE', 'SP_IPO_DATE', 'SP_DATE_INCORPORATED']

BAD_TICKERS     = {'-', 'delisted', '', 'nan', 'n.a.', 'na', 'none', 'null'}
SOURCE_PRIORITY = {'self': 0, 'guo': 1, 'duo': 2, 'ish': 3}
UNRESOLVED      = {'unresolvable_no_ticker', 'unresolvable_no_ownership'}
IN_UNIVERSE     = {'self', 'guo', 'duo', 'ish', 'identifier_fallback',
                   'delisted_kept'}

EUROPEAN_ISO = {'AT','BE','BG','HR','CY','CZ','DK','EE','FI','FR','DE','GR',
                'HU','IE','IT','LV','LT','LU','MT','NL','PL','PT','RO','SK',
                'SI','ES','SE','GB','NO','CH','IS','LI'}


In [21]:

# %%
def _usable_ticker(s):
    """True where a ticker field holds a real symbol (not a placeholder)."""
    return s.notna() & ~s.astype(str).str.strip().str.lower().isin(BAD_TICKERS)


def _norm_country(s):
    out = s.astype(str).str.strip().str.upper()
    return out.replace({'NAN': pd.NA, 'NONE': pd.NA, '': pd.NA})


def _prep_ciq(ciq_tickers, supplement_df=None):
    """CIQ lookup pool: one row per (ticker, country) so merges stay 1:1."""
    pool = ciq_tickers if supplement_df is None else pd.concat(
        [ciq_tickers, supplement_df], ignore_index=True)
    pool = pool[[c for c in CIQ_COLS if c in pool.columns]].copy()
    pool['SP_TICKER'] = pool['SP_TICKER'].astype(str).str.strip()
    pool['SP_COUNTRY_CODE'] = _norm_country(pool['SP_COUNTRY_CODE'])
    pool['_isin'] = pool['SP_ISIN'].notna().astype(int)
    return (pool.sort_values('_isin', ascending=False)
                .drop_duplicates(['SP_TICKER', 'SP_COUNTRY_CODE'])
                .drop(columns='_isin').reset_index(drop=True))


def _match_ciq(cand, pool):
    """
    Country-guarded ticker matching. Four passes on the remaining unmatched:
      A original ticker + country      C bare (pre-dot) + country
      B dot->space     + country       D bare, no country — ONLY for CIQ
                                         tickers unique across countries
    Row count preserved exactly.
    """
    cand = cand.reset_index(drop=True).copy()
    t = cand['cand_ticker'].astype(str).str.strip()
    cand['v_orig'], cand['v_space'] = t, t.str.replace('.', ' ', regex=False)
    cand['v_bare'] = t.str.split('.').str[0]
    cand['cand_country'] = _norm_country(cand['cand_country'])
    keep = list(cand.columns)

    uniq = pool.groupby('SP_TICKER')['SP_COUNTRY_CODE'].nunique()
    pool_nc = pool[pool['SP_TICKER'].isin(uniq[uniq == 1].index)]

    passes = [('A_orig_country', 'v_orig', True, pool),
              ('B_space_country', 'v_space', True, pool),
              ('C_bare_country', 'v_bare', True, pool),
              ('D_bare_nocountry', 'v_bare', False, pool_nc)]

    rest, out = cand, []
    for label, vcol, use_ctry, p in passes:
        if rest.empty:
            break
        lk = [vcol] + (['cand_country'] if use_ctry else [])
        rk = ['SP_TICKER'] + (['SP_COUNTRY_CODE'] if use_ctry else [])
        m = rest.merge(p, left_on=lk, right_on=rk, how='left', indicator=True)
        hit = m[m['_merge'] == 'both'].drop(columns='_merge')
        hit['match_pass'] = label
        out.append(hit)
        rest = m[m['_merge'] == 'left_only'][keep].reset_index(drop=True)

    if not rest.empty:
        for c in [c for c in CIQ_COLS if c in pool.columns]:
            rest[c] = pd.NA
        rest['match_pass'] = pd.NA
        out.append(rest)
    return pd.concat(out, ignore_index=True)


def _build_candidates(ets_core, ets_ownership):
    """Long table: one row per (entity, candidate parent). self + guo/duo/ish."""
    frames = []
    tcol = CORE_COLS['ticker']
    if tcol in ets_core.columns:
        m = _usable_ticker(ets_core[tcol])
        frames.append(pd.DataFrame({
            'bvd_id_number': ets_core.loc[m, 'bvd_id_number'].values,
            'cand_source':   'self',
            'cand_ticker':   ets_core.loc[m, tcol].values,
            'cand_country':  ets_core.loc[m, CORE_COLS['country']].values,
            'parent_bvd_id': ets_core.loc[m, 'bvd_id_number'].values,
            'parent_name':   ets_core.loc[m, CORE_COLS['name']].values,
        }))

    own = ets_ownership[
        (ets_ownership['bvd_id_number'] != ets_ownership['owner_bvd_id_number'])
        & _usable_ticker(ets_ownership['owner_ticker'])]
    if len(own):
        frames.append(pd.DataFrame({
            'bvd_id_number': own['bvd_id_number'].values,
            'cand_source':   own['owner_type'].str.lower().values,
            'cand_ticker':   own['owner_ticker'].values,
            'cand_country':  own['owner_country'].values,
            'parent_bvd_id': own['owner_bvd_id_number'].values,
            'parent_name':   own['owner_name'].values,
        }))

    cand = pd.concat(frames, ignore_index=True)
    cand['cand_priority'] = cand['cand_source'].map(SOURCE_PRIORITY)
    return cand


In [22]:
# %%
def build_ets_parent_map(ets_listing, ets_core, ets_ownership, ets_identifiers,
                         ciq_tickers, supplement_df, ticker_remap,
                         non_european):
    """Stage 1. One row per ETS entity, resolved to the best listed parent."""
    all_ids = (ets_listing[['bvd_id_number']].dropna()   # drop continuation rows
               .drop_duplicates().reset_index(drop=True))

    cand = _build_candidates(ets_core, ets_ownership)

    # confirmed manual ticker corrections, applied before matching
    if ticker_remap:
        bare = cand['cand_ticker'].astype(str).str.strip().str.split('.').str[0]
        for key, (tick, _exch, ctry) in ticker_remap.items():
            hit = bare == str(key).split('.')[0]
            cand.loc[hit, ['cand_ticker', 'cand_country']] = tick, ctry

    matched = _match_ciq(cand, _prep_ciq(ciq_tickers, supplement_df))

    # classify: resolved-European / non-European / unresolved
    ok = matched['match_pass'].notna()
    bare = matched['cand_ticker'].astype(str).str.split('.').str[0]
    non_eu = bare.isin(non_european) | (
        ok & matched['SP_COUNTRY_CODE'].notna()
        & ~matched['SP_COUNTRY_CODE'].isin(EUROPEAN_ISO))
    matched['outcome'] = np.select([ok & ~non_eu, non_eu],
                                   ['resolved_eu', 'non_european'],
                                   default='unresolved')
    matched['rank'] = matched['outcome'].map(
        {'resolved_eu': 0, 'non_european': 1, 'unresolved': 2})

    # collapse to best candidate per entity: outcome first, then self>guo>duo>ish
    best = (matched.sort_values(['bvd_id_number', 'rank', 'cand_priority'])
                   .drop_duplicates('bvd_id_number').copy())
    best['resolution_method'] = np.where(
        best['outcome'] == 'resolved_eu', best['cand_source'],
        best['outcome'].map({'non_european': 'non_european',
                             'unresolved': 'unresolvable_no_ticker'}))

    pm = best[['bvd_id_number', 'resolution_method', 'cand_source',
               'parent_bvd_id', 'parent_name', 'SP_TICKER', 'SP_ISIN',
               'SP_EXCHANGE', 'SP_COUNTRY_CODE', 'SP_COMPANY_STATUS',
               'SP_ENTITY_ID', 'match_pass']].rename(columns={
        'SP_TICKER': 'parent_ticker', 'SP_ISIN': 'parent_isin',
        'SP_EXCHANGE': 'parent_exchange', 'SP_COUNTRY_CODE': 'parent_country',
        'SP_COMPANY_STATUS': 'parent_status', 'SP_ENTITY_ID': 'parent_sp_id',
        'match_pass': 'matched_via'})

    # entities with no candidate at all -> LEI self-match, else unresolvable
    missing = set(all_ids['bvd_id_number']) - set(pm['bvd_id_number'])
    if missing:
        had_own = set(ets_ownership['bvd_id_number'])
        lei = (ets_identifiers[ets_identifiers['identifier_type'] == 'lei']
               .set_index('bvd_id_number')['identifier_value'])
        pool = _prep_ciq(ciq_tickers, supplement_df)
        by_lei = pool.dropna(subset=['SP_LEI']).drop_duplicates('SP_LEI')
        by_lei = by_lei.set_index('SP_LEI')

        rows = []
        for b in missing:
            r = {'bvd_id_number': b}
            v = lei.get(b)
            hit = by_lei.loc[[v]] if (v is not None and v in by_lei.index) else None
            if hit is not None and len(hit):
                h = hit.iloc[0]
                eu = pd.isna(h['SP_COUNTRY_CODE']) or h['SP_COUNTRY_CODE'] in EUROPEAN_ISO
                r.update({'resolution_method': 'identifier_fallback' if eu else 'non_european',
                          'cand_source': 'self', 'parent_bvd_id': b,
                          'parent_name': h.get('SP_ENTITY_NAME'),
                          'parent_ticker': h['SP_TICKER'], 'parent_isin': h['SP_ISIN'],
                          'parent_exchange': h.get('SP_EXCHANGE'),
                          'parent_country': h['SP_COUNTRY_CODE'],
                          'parent_status': h.get('SP_COMPANY_STATUS'),
                          'parent_sp_id': h.get('SP_ENTITY_ID'),
                          'matched_via': 'lei_fallback'})
            else:
                r['resolution_method'] = ('unresolvable_no_ticker' if b in had_own
                                          else 'unresolvable_no_ownership')
            rows.append(r)
        pm = pd.concat([pm, pd.DataFrame(rows)], ignore_index=True)

    pm = (pm.drop_duplicates('bvd_id_number').set_index('bvd_id_number')
            .reindex(all_ids['bvd_id_number']).reset_index())
    pm['resolution_method'] = pm['resolution_method'].fillna('unresolvable_no_ownership')
    pm['parent_delisting_date'] = pd.NaT
    pm['in_portfolio_universe'] = pm['resolution_method'].isin(IN_UNIVERSE)
    return pm


# %%
def flag_delisted_parents(pm, ets_ownership, ets_listing, eu_listing,
                          study_start=STUDY_START):
    """
    Stage 2. Orbis writes the literal 'Delisted' into owner_ticker with no
    date, so formerly-listed parents land in unresolvable_no_ticker next to
    genuinely never-listed ones. The owner's delisting date is already on disk.

    Only unresolvable_* entities are eligible — NOT ~in_portfolio_universe,
    which would resurrect deliberately-excluded non_european rows.
    """
    pm = pm.copy()
    cols = ['bvd_id_number', 'company_name', 'isin_number', 'ticker_symbol',
            'main_exchange', 'listing_status', 'delisting_date',
            'country_iso_code']
    lk = (pd.concat([ets_listing[cols], eu_listing[cols]], ignore_index=True)
            .dropna(subset=['bvd_id_number'])
            .drop_duplicates('bvd_id_number').set_index('bvd_id_number'))
    lk['delisting_date'] = pd.to_datetime(lk['delisting_date'], errors='coerce')
    lk['ticker_symbol'] = lk['ticker_symbol'].where(
        _usable_ticker(lk['ticker_symbol']))          # 'Delisted' -> NaN

    eligible = set(pm.loc[pm['resolution_method'].isin(UNRESOLVED), 'bvd_id_number'])
    is_del = ets_ownership['owner_ticker'].astype(str).str.strip().str.lower() == 'delisted'
    d = ets_ownership[is_del & ets_ownership['bvd_id_number'].isin(eligible)].copy()
    if d.empty:
        return pm

    d = d.join(lk, on='owner_bvd_id_number', rsuffix='_lk')
    d = (d.sort_values('delisting_date', ascending=False)
           .drop_duplicates('bvd_id_number').set_index('bvd_id_number'))

    ctry = _norm_country(d['country_iso_code'])
    d['method'] = np.select(
        [d['company_name'].isna(),                          # no lookup row
         ctry.notna() & ~ctry.isin(EUROPEAN_ISO),           # non-European parent
         d['delisting_date'].isna(),                        # listed once, no date
         d['delisting_date'] >= study_start],               # delisted in window
        ['delisted_unknown', 'non_european', 'delisted_undated', 'delisted_kept'],
        default='delisted_pre2018')

    hit = pm['bvd_id_number'].map(d['method']).notna()
    pm.loc[hit, 'resolution_method'] = pm.loc[hit, 'bvd_id_number'].map(d['method'])

    # overwrite ALL parent fields from the lookup — never leave Stage 1 residuals
    for pcol, lcol in [('parent_bvd_id', None), ('parent_name', 'company_name'),
                       ('parent_ticker', 'ticker_symbol'),
                       ('parent_isin', 'isin_number'),
                       ('parent_exchange', 'main_exchange'),
                       ('parent_country', 'country_iso_code'),
                       ('parent_status', 'listing_status'),
                       ('parent_delisting_date', 'delisting_date')]:
        src = (d.index.to_series() if lcol is None else d[lcol])
        pm.loc[hit, pcol] = pm.loc[hit, 'bvd_id_number'].map(
            d['owner_bvd_id_number'] if lcol is None else src)
    pm.loc[hit, 'parent_sp_id'] = pd.NA          # no CIQ id for these yet
    pm.loc[hit, 'matched_via'] = 'delisted_lookup'

    pm['in_portfolio_universe'] = pm['resolution_method'].isin(IN_UNIVERSE)
    return pm


def apply_european_backstop(pm):
    """Final guard: nothing outside Europe stays in the universe."""
    leak = (pm['in_portfolio_universe'] & pm['parent_country'].notna()
            & ~_norm_country(pm['parent_country']).isin(EUROPEAN_ISO))
    pm.loc[leak, ['resolution_method', 'in_portfolio_universe']] = 'non_european', False
    pm.loc[leak, 'parent_delisting_date'] = pd.NaT
    if leak.sum():
        print(f"  backstop removed {leak.sum()} non-European parents")
    return pm


def report(pm):
    print(f"\n{'='*52}\nETS Parent Map\n{'='*52}")
    print(f"  rows / expected: {len(pm):,} / {pm['bvd_id_number'].nunique():,}")
    print(pm['resolution_method'].value_counts(dropna=False).to_string())
    u = pm[pm['in_portfolio_universe']]
    print(f"\n  in universe:        {len(u):,}")
    print(f"  unique tickers:     {u['parent_ticker'].nunique():,}")
    print(f"  unique ISINs:       {u['parent_isin'].nunique():,}")
    print(f"  delisted (need ISIN pricing): "
          f"{int(pm['resolution_method'].eq('delisted_kept').sum()):,}")
    print(f"  countries:          "
          f"{sorted(u['parent_country'].dropna().unique().tolist())}")



In [23]:
TICKER_REMAP = {'CZG': ('COLT', 'SEP', 'CZ'),
 'RO': ('ROP', 'SWX', 'CH'),
 'ETEX': ('094124453', 'ENXTBR', 'BE'),
 'RY8': ('RY8', 'DUSE', 'DE'),
 'CTS1L': ('CTS', 'WSE', 'LT'),
 'ESSITY.A': ('ESSITY B', 'OM', 'SE'),
 'STW5': ('ACT', 'XTRA', 'DE')}

NON_EUROPEAN_TICKERS = {'000210',
 '000270',
 '005380',
 '01113',
 '02269',
 '034730',
 '051910',
 '161390',
 '361610',
 '373220',
 'ALPEKA',
 'BINANIIND',
 'BRKB',
 'ENPG',
 'HINDALCO',
 'JBFIND',
 'LBTYA',
 'MOTHERSUMI',
 'NEMAKA',
 'RUAL',
 'TATACHEM',
 'TATASTEEL',
 'TATYY',
 'TMPV',
 'VNTRF'}

In [24]:
# ── run ───────────────────────────────────────────────────────────────────
ets_parent_map = build_ets_parent_map(
    ets_listing, ets_core, ets_ownership, ets_identifiers,
    ciq_tickers   = load_raw('ciq_ets_owners_tickers_fetched.csv'),
    supplement_df = load_raw('ets_owner_supplement.csv'),
    ticker_remap  = TICKER_REMAP,
    non_european  = NON_EUROPEAN_TICKERS,
)
ets_parent_map = flag_delisted_parents(ets_parent_map, ets_ownership,
                                       ets_listing, eu_listing)

ets_parent_map = apply_european_backstop(ets_parent_map)
report(ets_parent_map)


ETS Parent Map
  rows / expected: 6,803 / 6,803
resolution_method
unresolvable_no_ticker       3939
unresolvable_no_ownership    1094
guo                          1019
non_european                  457
delisted_pre2018               81
ish                            73
delisted_kept                  62
self                           42
delisted_undated               15
duo                            11
delisted_unknown               10

  in universe:        1,207
  unique tickers:     335
  unique ISINs:       336
  delisted (need ISIN pricing): 62
  countries:          ['AT', 'BE', 'BG', 'CH', 'CZ', 'DE', 'DK', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LT', 'LU', 'NL', 'NO', 'PL', 'PT', 'RO', 'SE', 'SK']


In [25]:
kept = ets_parent_map[ets_parent_map['resolution_method'] == 'delisted_kept']

print(kept['parent_ticker'].notna().sum(), "have a ticker,",
      kept['parent_ticker'].isna().sum(), "are ISIN-only")

62 have a ticker, 0 are ISIN-only


### (3) Remove ETS Overlap from Non-ETS Universe



In [62]:
STUDY_START = pd.Timestamp('2018-01-01')
STUDY_END   = pd.Timestamp('2024-12-31')
 
# reuse from Section 2
# EUROPEAN_ISO, _usable_ticker, _norm_country
 
 
# %% 3a — collapse ETS entities to company level
u = ets_parent_map[ets_parent_map['in_portfolio_universe']].copy()

In [63]:
# security-level key: ISIN is what you trade (share classes are separate
# securities), fall back to ticker where Orbis gave no ISIN
u['company_id'] = u['parent_isin'].fillna(u['parent_ticker'])
if u['company_id'].isna().any():
    print(f"  ! {u['company_id'].isna().sum()} in-universe rows have neither "
          f"ISIN nor ticker — dropped")
    u = u[u['company_id'].notna()]

In [64]:
ets_companies = (u.groupby('company_id', dropna=False)
    .agg(company_name   = ('parent_name', 'first'),
         ticker         = ('parent_ticker', 'first'),
         isin           = ('parent_isin', 'first'),
         exchange       = ('parent_exchange', 'first'),
         country        = ('parent_country', 'first'),
         sp_entity_id   = ('parent_sp_id', 'first'),
         bvd_id_number  = ('parent_bvd_id', 'first'),
         listing_status = ('parent_status', 'first'),
         delisting_date = ('parent_delisting_date', 'max'),
         n_ets_entities = ('bvd_id_number', 'nunique'),
         resolution     = ('resolution_method',
                           lambda s: ','.join(sorted(set(s)))),
         ets_entity_ids = ('bvd_id_number',
                           lambda s: '|'.join(sorted(s.astype(str)))))
    .reset_index())

ets_companies['universe'] = 'ETS'

# Delisted companies are dropped: the deliverable is a live prediction system,
# which can only ever score currently-listed securities, and delisted names
# need manual CIQ pricing that does not scale. Counts are recorded here so the
# resulting survivorship bias can be quantified in the limitations section.
n_ets_delisted = int(ets_companies['delisting_date'].notna().sum())
ets_companies = ets_companies[ets_companies['delisting_date'].isna()].copy()

ets_companies['data_start_date'] = STUDY_START
ets_companies['data_end_date']   = STUDY_END

print(f"3a  ETS entities in universe : {len(u):,}")
print(f"    unique companies         : {len(ets_companies) + n_ets_delisted:,}")
print(f"    delisted, dropped        : {n_ets_delisted:,}")
print(f"    -> ETS sample            : {len(ets_companies):,}")
print(f"    top holders: ")
print(ets_companies.nlargest(5, 'n_ets_entities')[
    ['company_name', 'ticker', 'n_ets_entities']].to_string(index=False))
 

3a  ETS entities in universe : 1,207
    unique companies         : 336
    delisted, dropped        : 36
    -> ETS sample            : 300
    top holders: 
             company_name ticker  n_ets_entities
                    ENGIE   ENGI              62
COMPAGNIE DE SAINT-GOBAIN    SGO              41
     VEOLIA ENVIRONNEMENT    VIE              39
          WIENERBERGER AG    WIE              26
                  E.ON SE   EOAN              26


In [65]:
# %% 3b — listed European candidates from the EU universe
eu = eu_listing.copy()
eu['delisting_date'] = pd.to_datetime(eu['delisting_date'], errors='coerce')
eu['_ticker'] = eu['ticker_symbol'].where(_usable_ticker(eu['ticker_symbol']))
eu['_country'] = _norm_country(eu['country_iso_code'])

listed   = eu['_ticker'].notna() | eu['isin_number'].notna()
european = eu['_country'].isin(EUROPEAN_ISO)   # redundant (Orbis screen was
                                               # European) but kept as a guard
still_listed = eu['delisting_date'].isna()     # symmetric with the ETS rule

eu_candidates = eu[listed & european & still_listed].copy()

print(f"\n3b  eu_listing rows          : {len(eu):,}")
print(f"    listed                   : {listed.sum():,}")
print(f"    + European               : {(listed & european).sum():,}")
print(f"    delisted, dropped        : {(listed & european & ~still_listed).sum():,}")
print(f"    -> EU candidates         : {len(eu_candidates):,}")
 


3b  eu_listing rows          : 21,660
    listed                   : 15,346
    + European               : 15,346
    delisted, dropped        : 6,740
    -> EU candidates         : 8,606


In [66]:
# %% 3c — remove ETS overlap from the control group
# Three exclusion routes. A control firm must have NO carbon exposure via the
# EU ETS, so it is dropped if it is an ETS parent, is itself an ETS entity, or
# sits under an ETS parent in the ownership chain.
ets_isins   = set(ets_companies['isin'].dropna())
ets_tickers = set(ets_companies['ticker'].dropna())
ets_parents = set(ets_parent_map['parent_bvd_id'].dropna())
ets_entity  = set(ets_parent_map['bvd_id_number'].dropna())
ets_bvd_all = ets_parents | ets_entity

by_isin   = eu_candidates['isin_number'].isin(ets_isins)
by_ticker = eu_candidates['_ticker'].isin(ets_tickers)
by_bvd    = eu_candidates['bvd_id_number'].isin(ets_bvd_all)
by_chain  = (eu_candidates['guo_bvd_id_number'].isin(ets_bvd_all)
             | eu_candidates['duo_bvd_id_number'].isin(ets_bvd_all)
             | eu_candidates['ish_bvd_id_number'].isin(ets_bvd_all))

overlap = by_isin | by_ticker | by_bvd | by_chain
eu_control = eu_candidates[~overlap].copy()

print(f"\n3c  overlap removed by:")
print(f"      ISIN match             : {by_isin.sum():,}")
print(f"      ticker match           : {by_ticker.sum():,}")
print(f"      bvd_id (is an ETS co.) : {by_bvd.sum():,}")
print(f"      ownership chain        : {by_chain.sum():,}")
print(f"      any of the above       : {overlap.sum():,}")
print(f"    EU control remaining     : {len(eu_control):,}")

 


3c  overlap removed by:
      ISIN match             : 288
      ticker match           : 339
      bvd_id (is an ETS co.) : 434
      ownership chain        : 468
      any of the above       : 618
    EU control remaining     : 7,988


In [67]:
# %% 3d — stack into one master sample
eu_control_std = pd.DataFrame({
    'company_id':     eu_control['isin_number'].fillna(eu_control['_ticker']),
    'company_name':   eu_control['company_name'],
    'ticker':         eu_control['_ticker'],
    'isin':           eu_control['isin_number'],
    'exchange':       eu_control['main_exchange'],
    'country':        eu_control['_country'],
    'sp_entity_id':   pd.NA,
    'bvd_id_number':  eu_control['bvd_id_number'],
    'listing_status': eu_control['listing_status'],
    'delisting_date': eu_control['delisting_date'],
    'n_ets_entities': 0,
    'resolution':     'eu_direct',
    'ets_entity_ids': pd.NA,
    'universe':       'EU',
})

eu_control_std['data_start_date'] = STUDY_START
eu_control_std['data_end_date']   = STUDY_END
 

In [68]:
# one row per security across BOTH universes; ETS wins any residual tie
sample_master = (pd.concat([ets_companies, eu_control_std], ignore_index=True)
                   .dropna(subset=['company_id'])
                   .sort_values('universe')          # ETS before EU
                   .drop_duplicates('company_id', keep='first')
                   .reset_index(drop=True))

print(f"\n3d  sample_master           : {len(sample_master):,} companies")
print(sample_master['universe'].value_counts().to_string())
no_ticker = sample_master['ticker'].isna().sum()
print(f"    have a ticker (yFinance) : {sample_master['ticker'].notna().sum():,}")
print(f"    ISIN only, no ticker     : {no_ticker:,}   <- cannot yFinance")
print(f"    ETS : EU ratio           : 1 : "
      f"{len(sample_master[sample_master.universe=='EU']) / max(len(sample_master[sample_master.universe=='ETS']),1):.1f}")

sample_master.to_csv('sample_master.csv', index=False)
sample_master.head()


3d  sample_master           : 8,288 companies
universe
EU     7988
ETS     300
    have a ticker (yFinance) : 8,179
    ISIN only, no ticker     : 109   <- cannot yFinance
    ETS : EU ratio           : 1 : 26.6


,company_id,company_name,ticker,isin,exchange,country,sp_entity_id,bvd_id_number,listing_status,delisting_date,n_ets_entities,resolution,ets_entity_ids,universe,data_start_date,data_end_date
0,AT000000STR1,STRABAG SE,STR,AT000000STR1,WBAG,AT,4995608,AT9090003478,Operating,NaT,2,guo,DE5190697268|HU26185059,ETS,2018-01-01,2024-12-31
1,IE00BYTBXV33,RYANAIR HOLDINGS PLC,RYA,IE00BYTBXV33,ISE,IE,4994438,IE249885,Operating,NaT,3,guo,AT9110409727|IE104547|PL367803358,ETS,2018-01-01,2024-12-31
2,IE00B010DT83,C&C GROUP PLC,CCR,IE00B010DT83,LSE,IE,4915001,IE383466,Operating,NaT,2,guo,GB07063165|GBSC362352,ETS,2018-01-01,2024-12-31
3,IE0004927939,KINGSPAN GROUP PLC,KRX,IE0004927939,ISE,IE,4296509,IE070576,Operating,NaT,3,guo,FR347517930|FR383772571|PL301909810,ETS,2018-01-01,2024-12-31
4,IE0004906560,KERRY GROUP PLC,KRZ,IE0004906560,ISE,IE,4229009,IE111471,Operating,NaT,2,guo,GB00329695|IE206782,ETS,2018-01-01,2024-12-31


#### linking trucost emissions data to EU companies

In [69]:
# %% Attach an in-universe ISIN to each GHG company-year row

eu_ghg_raw = load_raw('eu_ghg_final.csv')
compustat_id = load_raw('compustat_identifier_match.csv')

universe_isins = set(sample_master.loc[sample_master['universe'] == 'EU', 'isin'])

bridge = compustat_id.copy()   # keep all ISINs

bridge['in_universe'] = bridge['isin'].isin(universe_isins)
bridge = (bridge.sort_values(['companyid', 'in_universe'], ascending=[True, False])
                .drop_duplicates('companyid', keep='first'))   # dedup on MERGE key
 
# sanity: must be 0 or the merge will still inflate
assert bridge['companyid'].duplicated().sum() == 0
 
eu_ghg = eu_ghg_raw.merge(bridge[['companyid', 'isin', 'in_universe']],left_on='Company ID',
                          right_on='companyid', how='left')
 
assert len(eu_ghg) == len(eu_ghg_raw), "row count changed — bridge still has dupes"
 
print(f"GHG rows          : {len(eu_ghg):,}")
print(f"got an ISIN       : {eu_ghg['isin'].notna().sum():,}")
print(f"ISIN in universe  : {eu_ghg['in_universe'].fillna(False).sum():,}")
print(f"unique companies in universe: "
      f"{eu_ghg.loc[eu_ghg['in_universe'] == True, 'isin'].nunique():,}")

GHG rows          : 12,790
got an ISIN       : 12,790
ISIN in universe  : 9,550
unique companies in universe: 1,007


In [70]:
# how does coverage split across ETS vs EU control?
covered = set(eu_ghg.loc[eu_ghg['in_universe']==True, 'isin'])
sample_master['has_emissions'] = sample_master['isin'].isin(covered)
print(sample_master.groupby('universe')['has_emissions'].agg(['sum','count']))

# year span
print(eu_ghg.loc[eu_ghg['in_universe']==True, 'Year'].agg(['min','max']))  # adjust col name

           sum  count
universe             
ETS          0    300
EU        1007   7988
min    2013
max    2025
Name: Year, dtype: int64


### (4) Creation of database



Load the resolved tables into SQLite

Creates the database from schema.sql and loads what exists now: master_company_list, ets_entity, the entity->company link, and the decomposed Orbis tables. Price / fundamental / emissions loaders come later as those pulls complete.

In [71]:
import pandas as pd
from src.db import init_db, load_table   

In [72]:
DB     = '/Users/admin/Desktop/carbon-portfolio-project-v2/data/carbon.db'
SCHEMA = '/Users/admin/Desktop/carbon-portfolio-project-v2/sql/schema.sql'

In [73]:
# %%
con = init_db(db_path=DB, schema_path=SCHEMA, drop_existing=True)
print("Loading:")

Loading:


In [74]:
load_table(con, sample_master, 'master_company_list')

  master_company_list      8,288 rows   (no data for: sector)


In [75]:
load_table(con, ets_parent_map, 'ets_entity',
           colmap={'company_name': 'company_name'})

  ets_entity               6,803 rows   (no data for: company_name, country)


In [76]:
ets_rows = sample_master[sample_master['universe'] == 'ETS']
links = (ets_rows[['company_id', 'ets_entity_ids']]
         .dropna(subset=['ets_entity_ids'])
         .assign(bvd_id_number=lambda d: d['ets_entity_ids'].str.split('|'))
         .explode('bvd_id_number')[['bvd_id_number', 'company_id']]
         .drop_duplicates())
load_table(con, links, 'ets_entity_company')

  ets_entity_company       1,145 rows


In [77]:
orbis_core_all = pd.concat([
    ets_core.assign(source_table='ets'),
    eu_core.assign(source_table='eu'),
], ignore_index=True).drop_duplicates('bvd_id_number')

In [78]:
load_table(con, orbis_core_all, 'orbis_core', colmap={
    'nace_rev_2_core_code_4_digits': 'nace_code',
    'nace_rev_2_core_code_description': 'nace_description',
    'standardized_legal_form': 'legal_form',
    'date_of_incorporation': 'incorporation_date',
    'operating_revenue_turnover_th_usd_last_avail_yr': 'operating_revenue',
    'number_of_employees_last_avail_yr': 'n_employees',
    'last_avail_year': 'last_avail_year',
})

  orbis_core              26,521 rows


In [79]:
load_table(con, pd.concat([ets_identifiers, eu_identifiers], ignore_index=True)
             .drop_duplicates(['bvd_id_number', 'identifier_type']),
           'orbis_identifiers')

  orbis_identifiers       80,022 rows


In [80]:
own = pd.concat([ets_ownership, eu_ownership], ignore_index=True)
own['owner_type'] = own['owner_type'].str.lower()
load_table(con, own.drop_duplicates(['bvd_id_number', 'owner_type']),
           'orbis_ownership')


  orbis_ownership         49,323 rows


In [82]:
# %% verify
print("\nRow counts:")
for t in ['master_company_list', 'ets_entity', 'ets_entity_company',
          'orbis_core', 'orbis_identifiers', 'orbis_ownership']:
    n = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f"  {t:22s} {n:>7,}")
 
print("\nUniverse split:")
print(pd.read_sql(
    'SELECT universe, COUNT(*) n FROM master_company_list GROUP BY universe', con))
 
print("\nTop ETS companies by entity count:")
print(pd.read_sql("""
    SELECT m.company_name, m.ticker, COUNT(e.bvd_id_number) AS n_entities
    FROM master_company_list m
    JOIN ets_entity_company e ON e.company_id = m.company_id
    GROUP BY m.company_id
    ORDER BY n_entities DESC
    LIMIT 5
""", con))


Row counts:
  master_company_list      8,288
  ets_entity               6,803
  ets_entity_company       1,145
  orbis_core              26,521
  orbis_identifiers       80,022
  orbis_ownership         49,323

Universe split:
  universe     n
0      ETS   300
1       EU  7988

Top ETS companies by entity count:
                company_name ticker  n_entities
0                      ENGIE   ENGI          62
1  COMPAGNIE DE SAINT-GOBAIN    SGO          41
2       VEOLIA ENVIRONNEMENT    VIE          39
3            WIENERBERGER AG    WIE          26
4                    E.ON SE   EOAN          26


In [91]:
# integrity: every link should resolve on both sides

orphan = con.execute("""
    SELECT COUNT(*) FROM ets_entity_company e
    LEFT JOIN master_company_list m ON m.company_id = e.company_id
    WHERE m.company_id IS NULL
""").fetchone()[0]

print(f"\nOrphaned links: {orphan}   (expected 0)")


Orphaned links: 0   (expected 0)


In [84]:
# --- backfill sector from orbis_core via the uniform bvd link ----------------
# ---- Note: master_company_list already loaded, simply updating the table with sector information:
con.execute("""
    UPDATE master_company_list
    SET sector = (SELECT c.bvd_sectors FROM orbis_core c
                  WHERE c.bvd_id_number = master_company_list.bvd_id_number)
    WHERE bvd_id_number IS NOT NULL
""")
con.commit()

In [85]:
# report FK coverage: EU should be ~complete, ETS partial (CIQ-only parents)
print("\nbvd_id_number -> orbis_core coverage:")
print(pd.read_sql("""
    SELECT m.universe,
           COUNT(*)                                             AS companies,
           SUM(m.bvd_id_number IS NOT NULL)                     AS has_bvd,
           SUM(c.bvd_id_number IS NOT NULL)                     AS resolves_in_core,
           SUM(m.sector IS NOT NULL)                            AS has_sector
    FROM master_company_list m
    LEFT JOIN orbis_core c ON c.bvd_id_number = m.bvd_id_number
    GROUP BY m.universe
""", con).to_string(index=False))
 
 


bvd_id_number -> orbis_core coverage:
universe  companies  has_bvd  resolves_in_core  has_sector
     ETS        300      300               297         296
      EU       7988     7988              7988        7934


In [86]:
# %%
# %% Load Trucost (non-ETS) Scope 1 into company_emissions
# Run after ghg_isin_merge.py (produces eu_ghg) and the DB is initialised.
 
trucost = (eu_ghg[eu_ghg['in_universe'] == True]
           .rename(columns={'isin': 'company_id',
                            'Year': 'year',
                            'Absolute GHG Scope 1': 'scope1_emissions'})
           .loc[:, ['company_id', 'year', 'scope1_emissions']]
           .dropna(subset=['company_id', 'year'])
           .drop_duplicates(['company_id', 'year']))       # one row per company-year
trucost['source'] = 'trucost'
 
load_table(con, trucost, 'company_emissions')

  company_emissions        9,448 rows


In [87]:
# backfill the coverage flag
con.execute("""
    UPDATE master_company_list
    SET has_emissions = (SELECT COUNT(*) > 0 FROM company_emissions ce
                         WHERE ce.company_id = master_company_list.company_id)
""")
con.commit()

In [88]:
print(pd.read_sql("""
    SELECT m.universe, ce.source, COUNT(DISTINCT ce.company_id) AS companies,
           COUNT(*) AS company_years, MIN(ce.year) AS y0, MAX(ce.year) AS y1
    FROM company_emissions ce
    JOIN master_company_list m ON m.company_id = ce.company_id
    GROUP BY m.universe, ce.source
""", con).to_string(index=False))

universe  source  companies  company_years   y0   y1
      EU trucost       1007           9448 2013 2025


In [90]:
# the 4 sanity views
pd.read_sql("SELECT * FROM v_universe_summary LIMIT 5", con)
# emissions actually joins to companies
pd.read_sql("""SELECT m.universe, COUNT(DISTINCT ce.company_id) n
               FROM company_emissions ce JOIN master_company_list m
               ON m.company_id=ce.company_id GROUP BY m.universe""", con)

,universe,n
0,EU,1007


### (4) Filtering Companies (not completed)

**Filter criteria**

* Geography


Country of incorporation: EU27 + UK + Norway + Switzerland (keeps your "European equity markets" framing clean and matches EUA price exposure geography). 
Listed on a major European exchange — use a whitelist approach (Euronext, LSE, Deutsche Börse, SIX, Nasdaq Nordic, Borsa Italiana, BME, etc.) rather than just country, since some European firms list primarily in the US

* Listing status & data availability

Active OR delisted-after-2018 (same survivorship bias logic you applied to ETS accounts — include firms that were listed during your study period even if now gone)
IPO date before 2021 (gives at least 3 years of data within your 2018–2024 window)
Primary listing only — exclude secondary listings and depositary receipts to avoid duplicates

* Size & liquidity

Exclude penny stocks: minimum share price threshold (e.g. >€0.50 average) or minimum market cap (e.g. >€50m) — exact cutoff is somewhat arbitrary, just document it
OTC / pink sheet listings excluded

* Sector

Exclude financials (GICS 40) and real estate (GICS 60) — standard in cross-sectional return studies, capital structure is fundamentally different
Utilities (GICS 55): this one is worth flagging. Many utilities are ETS-regulated, so they'll get caught by your ETS filter anyway. For non-ETS utilities that slipped through, excluding them is defensible but not mandatory — just be consistent and document it

* Fundamentals availability

At least 3 years of non-missing revenue/total assets from your S&P Capital IQ export (you can check this programmatically after the price pull rather than filtering in Orbis)